# Packed backbone inputs
This notebook runs fully offline by default: a local toy tokenizer and a tiny random Qwen3 decoder check structure and numerical invariants. They do not measure pretrained quality or 4B memory usage. The final pretrained section is explicitly opt-in.
Run `uv sync --group notebooks` and select this project's `.venv` kernel.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import copy
import time
import torch
import matplotlib.pyplot as plt
from tokenizers import Tokenizer, models, pre_tokenizers, trainers
from transformers import PreTrainedTokenizerFast, Qwen3Config, Qwen3Model
from src.data.schema import Example, Question, Option, Target
from src.backbone.inputs import InputBuilder, InputConfig, build_mask
from src.backbone.model import BackboneConfig, FrozenBackbone
from src.backbone.features import PoolingConfig, extract_features
torch.manual_seed(42)
torch.set_num_threads(2)


## Local examples and tokenizer
Two contexts, three questions and seven options exercise unequal batch dimensions. Target values are stored in the examples but never serialized into backbone input.

In [ ]:
def question(identity, text, choices):
    return Question(identity, 0, text, 'demo', 'choice',
                    [Option(str(i), choice) for i, choice in enumerate(choices)],
                    Target('hard', [1.0] + [0.0] * (len(choices) - 1)))
examples = [
    Example('first', 'Alice lives in Lyon. Bob lives in Paris.', [
        question('city', 'Where does Alice live?', ['Lyon', 'Paris']),
        question('person', 'Who lives in Paris?', ['Bob', 'Alice', 'Nobody']),
    ]),
    Example('second', 'The light is blue.', [
        question('color', 'What color is the light?', ['blue', 'red']),
    ]),
]
examples[0].questions[0].options[0].criteria = 'Use the stated city.'
backend = Tokenizer(models.WordLevel(unk_token='[UNK]'))
backend.pre_tokenizer = pre_tokenizers.Whitespace()
texts = ['Context: Question: Option: Criteria: Decision: <user> </user>']
for example in examples:
    texts.append(example.context)
    for q in example.questions:
        texts.append(q.text)
        texts.extend(o.text + ' ' + (o.criteria or '') for o in q.options)
backend.train_from_iterator(texts, trainers.WordLevelTrainer(special_tokens=['[UNK]', '[PAD]', '[EOS]']))
tokenizer = PreTrainedTokenizerFast(tokenizer_object=backend, unk_token='[UNK]',
                                    pad_token='[PAD]', eos_token='[EOS]')
tokenizer.chat_template = '{{ "<user>\n" + messages[0]["content"] + "\n</user>" }}'
modes = ['causal', 'independent_questions', 'independent_options']


## Three attention modes and their positions
Rows are queries, columns are keys. Bright cells are permitted attention. Each Decision block reads all options of its own question; option branches themselves remain isolated in independent_options.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 7))
for column, mode in enumerate(modes):
    batch = InputBuilder(tokenizer, InputConfig(attention=mode, query_source='decision'))(examples)
    n = batch.lengths[0]
    # Materialize causal attention only for this visualization.
    visible = build_mask(batch.samples[0].blocks, n, mode)
    axes[0, column].imshow(visible, origin='upper', interpolation='nearest')
    axes[0, column].set_title(mode)
    axes[0, column].set_xlabel('Key token')
    axes[0, column].set_ylabel('Query token')
    axes[1, column].plot(batch.position_ids[0, :n].numpy())
    axes[1, column].set_xlabel('Physical token index')
    axes[1, column].set_ylabel('Position ID')
    for block in batch.samples[0].blocks:
        axes[0, column].axhline(block.start - .5, color='red', lw=.4)
        axes[0, column].axvline(block.start - .5, color='red', lw=.4)
        axes[1, column].axvline(block.start, color='gray', lw=.4)
plt.tight_layout()
plt.show()
for block in batch.samples[0].blocks:
    print(block.role, 'q=', block.question, 'o=', block.option,
          'tokens=', (block.start, block.stop), 'content=', (block.content_start, block.content_stop),
          'positions=', batch.samples[0].positions[block.start:block.stop])


## Offline decoder forward and feature shapes
Layer 1 is a decoder-block output. The last layer (3 here) is after final RMS normalization. Only requested depths are retained.

In [ ]:
tiny_config = Qwen3Config(vocab_size=len(tokenizer), hidden_size=48, intermediate_size=96,
                          num_hidden_layers=3, num_attention_heads=4, num_key_value_heads=2,
                          head_dim=12, max_position_embeddings=512, attention_dropout=0.0,
                          pad_token_id=tokenizer.pad_token_id)
tiny_config._attn_implementation = 'sdpa'
tiny = FrozenBackbone.from_model(Qwen3Model(tiny_config))
builder = InputBuilder(tokenizer)
batch = builder(examples)
output = tiny.forward(batch, layers=(1, -1))
features = extract_features(output, batch, PoolingConfig(include_context=True))
print('Input:', batch.input_ids.shape, 'mask:', batch.allowed_attention.shape)
print('Questions:', features.questions.shape, 'options:', features.options.shape)
print('Option -> question:', features.option_to_question.tolist())
print('Question -> context:', features.question_to_context.tolist())
print('Context tokens:', [tuple(context.shape) for context in features.contexts])
assert features.questions.shape == (3, 2, 48)
assert features.options.shape == (7, 2, 48)


## Packed vs separate causal forwards
For each question/option endpoint, gather its visible path and run a separate ordinary causal forward with sequential positions. Decision aggregation has multiple branches, so it is checked separately below.

In [ ]:
with torch.no_grad():
    for mode in modes:
        packed = InputBuilder(tokenizer, InputConfig(attention=mode))(examples)
        actual = tiny.forward(packed).hidden_states[3]
        for b, sample in enumerate(packed.samples):
            for block in sample.blocks:
                if block.role not in {'question', 'option'}:
                    continue
                endpoint = block.content_stop - 1
                keep = (torch.arange(endpoint + 1) if mode == 'causal' else
                        packed.allowed_attention[b, 0, endpoint].nonzero().flatten())
                ids = packed.input_ids[b:b+1, keep]
                reference = tiny.load()(input_ids=ids, position_ids=torch.arange(len(keep))[None],
                                        use_cache=False).last_hidden_state[0, -1]
                torch.testing.assert_close(actual[b, endpoint], reference, atol=2e-5, rtol=2e-5)
        print(mode, 'separate causal forwards: OK')


## Isolation, Decision aggregation, padding, chat and pooling
Changing one option must not affect another option in independent_options. Independent question branches must match isolated single-question inputs, including Decision. Ordinary causal mode intentionally permits cross-question influence.

In [ ]:
for mode in modes:
    for fmt in ['plain', 'chat']:
        for query_source in ['question', 'decision']:
            config = InputConfig(attention=mode, prompt_format=fmt, query_source=query_source)
            builder = InputBuilder(tokenizer, config)
            packed = builder(examples)
            result = tiny.forward(packed)
            for pooling in ['last', 'mean']:
                pooled = extract_features(result, packed, PoolingConfig(question=pooling, option=pooling))
                assert pooled.questions.shape == (3, 1, 48)
                assert torch.isfinite(pooled.options).all()
            single = builder(examples[0])
            single_features = extract_features(tiny.forward(single), single)
            packed_features = extract_features(result, packed)
            torch.testing.assert_close(single_features.questions, packed_features.questions[:2], atol=2e-5, rtol=2e-5)
            if mode != 'causal':
                for q in range(2):
                    isolated = builder(Example('isolated', examples[0].context, [examples[0].questions[q]]))
                    isolated_features = extract_features(tiny.forward(isolated), isolated)
                    torch.testing.assert_close(isolated_features.questions[0], packed_features.questions[q], atol=2e-5, rtol=2e-5)
print('All modes, formats, query sources and pooling choices: OK')
builder = InputBuilder(tokenizer)
original = builder(examples)
before = extract_features(tiny.forward(original), original)
changed_examples = copy.deepcopy(examples)
changed_examples[0].questions[0].options[0].text = 'blue red Nobody'
changed = builder(changed_examples)
after = extract_features(tiny.forward(changed), changed)
torch.testing.assert_close(before.options[1:], after.options[1:], atol=2e-5, rtol=2e-5)
torch.testing.assert_close(before.questions, after.questions, atol=2e-5, rtol=2e-5)
print('Independent option mutation: OK')
try:
    InputBuilder(tokenizer, InputConfig(max_packed_tokens=4))(examples[0])
except ValueError as error:
    print('Explicit length rejection:', error)
else:
    raise AssertionError('Oversized input was not rejected')


## Optional pretrained backbone — disabled by default
Choose a checkpoint and ideally a pinned revision before enabling this cell. Qwen below is only a usage example, not the selected research backbone. Tokenizer access fetches configuration/tokenizer files; forward fetches missing weights and loads NF4 on CUDA. Nothing is downloaded while RUN_PRETRAINED is false. Real 4-bit tolerances and memory must be measured separately from the tiny float32 checks above.

In [ ]:
RUN_PRETRAINED = False
pretrained = FrozenBackbone(BackboneConfig(
    model_id='Qwen/Qwen3-4B-Instruct-2507', revision=None,
    cache_dir=ROOT / 'artifacts/models/hub',
))
assert not pretrained.loaded
if RUN_PRETRAINED:
    real_batch = InputBuilder(pretrained.tokenizer, InputConfig(max_packed_tokens=2048))(examples)
    pretrained.load()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    started = time.perf_counter()
    real_output = pretrained.forward(real_batch)
    torch.cuda.synchronize()
    real_features = extract_features(real_output, real_batch)
    print('Forward seconds:', time.perf_counter() - started)
    print('Peak reserved GiB:', torch.cuda.max_memory_reserved() / 2**30)
    print('Features:', real_features.questions.shape, real_features.options.shape)
    print(real_output.metadata)
else:
    print('Pretrained loading skipped; no Hub access or weight download.')


## Prepared dataset lengths — tokenizer only, opt-in
Set LENGTH_MODEL_ID and optionally a pinned LENGTH_REVISION, then enable RUN_LENGTHS. This reads all prepared splits using the actual input serializer. Only configuration/tokenizer files may be fetched; no model weights are loaded. Match LENGTH_INPUTS to the intended feature cache. Budget exceedance counts every question in an oversized shared-context record, because the record is processed as a whole. The budget is bypassed for inspection only; normal cache extraction still rejects oversized inputs.
The report includes context-weighted token quantiles, question-weighted budget coverage and category/task breakdowns. It is saved to artifacts/data/analysis/lengths.json. It does not choose truncation or filtering policies automatically.

In [ ]:
import json
from dataclasses import asdict
from collections import Counter, defaultdict
import numpy as np
import pyarrow.parquet as pq
from tqdm.auto import tqdm
from src.data.dataset import read_examples

RUN_LENGTHS = False
LENGTH_MODEL_ID = ''  # Choose the actual research checkpoint.
LENGTH_REVISION = None
LENGTH_DATASET = ROOT / 'artifacts/data/prepared/research_10_1_1'
LENGTH_INPUTS = InputConfig(attention='causal', prompt_format='plain', query_source='question')
BUDGETS = (2048, 4096, 8192)

def length_summary(values):
    return dict(zip(['median', 'p90', 'p95', 'p99', 'max'],
                    map(float, np.percentile(values, [50, 90, 95, 99, 100]))))

def coverage(rows):
    total = sum(row['questions'] for row in rows)
    return {str(budget): {
        'oversized_contexts': sum(row['tokens'] > budget for row in rows),
        'affected_questions': sum(row['questions'] for row in rows if row['tokens'] > budget),
        'affected_question_percent': 100 * sum(row['questions'] for row in rows if row['tokens'] > budget) / total,
    } for budget in BUDGETS}

if RUN_LENGTHS:
    if not LENGTH_MODEL_ID:
        raise ValueError('Set LENGTH_MODEL_ID before running the length analysis')
    length_backbone = FrozenBackbone(BackboneConfig(
        model_id=LENGTH_MODEL_ID, revision=LENGTH_REVISION,
        cache_dir=ROOT / 'artifacts/models/hub',
    ))
    length_builder = InputBuilder(length_backbone.tokenizer, LENGTH_INPUTS)
    source_manifest = json.loads((LENGTH_DATASET / 'manifest.json').read_text())
    report = {'model_id': LENGTH_MODEL_ID, 'revision': length_backbone.describe()['revision'],
              'inputs': asdict(LENGTH_INPUTS), 'dataset': str(LENGTH_DATASET), 'splits': {}}
    all_rows = []
    for split in ['train', 'validation', 'test']:
        source = LENGTH_DATASET / source_manifest['splits'][split]['file']
        total = pq.ParquetFile(source).metadata.num_rows
        rows, groups = [], {'category': defaultdict(list), 'task': defaultdict(list)}
        for example in tqdm(read_examples(source), total=total, desc=f'Lengths: {split}', unit='context'):
            encoded = length_builder.encode(example, check_length=False)
            context = next(block for block in encoded.blocks if block.role == 'context')
            row = {'tokens': len(encoded.input_ids), 'questions': len(example.questions),
                   'context_tokens': context.content_stop - context.content_start,
                   'position_extent': max(encoded.positions) + 1}
            rows.append(row)
            # Count each context once per category/task, with its question weight.
            for field in groups:
                counts = Counter(getattr(question, field) for question in example.questions)
                for label, count in counts.items():
                    groups[field][label].append({'tokens': row['tokens'], 'questions': count})
        split_report = {
            'contexts': len(rows), 'questions': sum(row['questions'] for row in rows),
            'packed_tokens': length_summary([row['tokens'] for row in rows]),
            'context_tokens': length_summary([row['context_tokens'] for row in rows]),
            'position_extent': length_summary([row['position_extent'] for row in rows]),
            'budgets': coverage(rows),
            'by_category': {key: coverage(value) for key, value in sorted(groups['category'].items())},
            'by_task': {key: coverage(value) for key, value in sorted(groups['task'].items())},
        }
        report['splits'][split] = split_report
        all_rows.extend(rows)
        print(split, json.dumps({key: value for key, value in split_report.items()
                                 if not key.startswith('by_')}, indent=2))
        for field in ['by_category', 'by_task']:
            for label, budgets in split_report[field].items():
                print(field, label, ' | '.join(
                    f"{budget}: {stats['affected_questions']} questions ({stats['affected_question_percent']:.1f}%)"
                    for budget, stats in budgets.items()))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist([row['tokens'] for row in all_rows], bins=80)
    axes[0].set(xlabel='Packed tokens per context record', ylabel='Contexts', yscale='log')
    axes[1].hist([row['context_tokens'] for row in all_rows], bins=80)
    axes[1].set(xlabel='Context content tokens', ylabel='Contexts', yscale='log')
    plt.tight_layout()
    plt.show()
    report_path = ROOT / 'artifacts/data/analysis/lengths.json'
    report_path.parent.mkdir(parents=True, exist_ok=True)
    report_path.write_text(json.dumps(report, indent=2) + '\n')
    print('Saved:', report_path, '| Backbone weights loaded:', length_backbone.loaded)
else:
    print('Length analysis skipped. Set the checkpoint and RUN_LENGTHS=True to run.')
